# Lesson 5. Gates and circuits

**Goal:** understand how the state changes between preparation and measurement: why gates are unitary matrices, what they do on the Bloch sphere, and how this relates to physics.

## 1. A gate is a unitary matrix

A gate takes a state $v$ and returns $Uv$. Quantum mechanics imposes two conditions:

1. **Linearity:** $U(\alpha e_0 + \beta e_1) = \alpha Ue_0 + \beta Ue_1$ (the superposition principle). The columns of the matrix are the images of the basis vectors.
2. **Norm preservation:** $\|Uv\| = 1$ for every state, otherwise probabilities would not add up to one.

A linear map preserving the norm of every vector is a **unitary matrix**: $U^{*}U = I$. It follows that:

- every gate is **reversible**: $U^{-1} = U^{*}$ (unlike classical AND);
- a gate preserves all inner products, $\langle Uu, Uv\rangle = \langle u, v\rangle$, so orthogonal states stay orthogonal and an orthonormal basis goes to an orthonormal basis.

**Unitarity does not depend on the basis:** changing orthonormal basis is $U \mapsto V^{*}UV$ with unitary $V$, and a product of unitaries is unitary. A gate preserves the norm in every orthonormal basis, because the norm is a property of the vector, not of its coordinates.

Measurement is **not** a gate: it is neither unitary nor reversible. It is usually a separate, final step (mid-circuit measurement also exists, e.g. for active reset and error correction).

## 2. Three basic gates

$$X = \begin{pmatrix}0 & 1\\ 1 & 0\end{pmatrix}, \qquad Z = \begin{pmatrix}1 & 0\\ 0 & -1\end{pmatrix}, \qquad H = \frac{1}{\sqrt2}\begin{pmatrix}1 & 1\\ 1 & -1\end{pmatrix}.$$

- **X** (quantum NOT): $X(\alpha, \beta) = (\beta, \alpha)$, swaps the amplitudes.
- **Z** (phase flip): $Z(\alpha, \beta) = (\alpha, -\beta)$. It does not change the moduli, so $P(0)$ and $P(1)$ stay the same; it changes the relative phase by $\pi$: $Zu_+ = u_-$.
- **H** (Hadamard): $He_0 = u_+$, $He_1 = u_-$, it swaps the $z$ basis with the $x$ basis. It is a normalised Hadamard matrix: without the factor $1/\sqrt2$ it would not be unitary.

## 3. On the Bloch sphere a gate is a rotation

A gate preserves inner products, so on the sphere it neither squeezes nor stretches: it is a **rotation**.

| Gate | Rotation | Effect |
|---|---|---|
| X | by $\pi$ about the $x$ axis | the poles swap, $u_\pm$ stay |
| Z | by $\pi$ about the $z$ axis | $u_+ \leftrightarrow u_-$ along the equator, poles stay |
| H | by $\pi$ about the axis $(x + z)/\sqrt2$ | $e_0 \leftrightarrow u_+$, $e_1 \leftrightarrow u_-$ |

Formally this is the homomorphism $SU(2) \to SO(3)$, a double cover (hence half-angles again).

**Measuring in another basis:** hardware measures only in $\{e_0, e_1\}$. To measure in $\{u_+, u_-\}$ you apply $H$ first: $Hu_+ = e_0$, $Hu_- = e_1$, so outcome 0 means "+" and 1 means "−".

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
GREEN, DARK, LIGHT, GREY, ORANGE = "#44B749", "#1A8A1F", "#96E199", "#8A8A8A", "#E08A00"

e0 = np.array([1, 0], dtype=complex)   # label |0⟩
e1 = np.array([0, 1], dtype=complex)   # label |1⟩

def norm(v):
    return np.sqrt(np.sum(np.abs(v) ** 2))

def show(v, n=None):
    """Print a state vector with bit labels (label = position written in binary)."""
    v = np.asarray(v)
    n = n or int(np.log2(len(v)))
    for k, a in enumerate(v):
        if abs(a) > 1e-12:
            print(f"  position {k} ({format(k, f'0{n}b')}): amplitude {np.round(a, 3)}, P = {abs(a)**2:.3f}")

X = np.array([[0, 1], [1, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
I2 = np.eye(2)
up, um = (e0 + e1) / np.sqrt(2), (e0 - e1) / np.sqrt(2)

for name, U in (("X", X), ("Z", Z), ("H", H)):
    print(f"{name}: unitary? {np.allclose(U.conj().T @ U, I2)}")

print("Z u+ =", np.round(Z @ up, 3), "= u-")
print("H Z H =\n", np.round((H @ Z @ H).real, 3) + 0.0, "\n(compare with the matrix X)")

X: unitary? True
Z: unitary? True
H: unitary? True
Z u+ = [ 0.707+0.j -0.707+0.j] = u-
H Z H =
 [[0. 1.]
 [1. 0.]] 
(compare with the matrix X)

## 4. Interference in action: $HH = I$

$$H(He_0) = Hu_+ = \frac{1}{\sqrt2}(He_0 + He_1) = \frac12\big[(e_0 + e_1) + (e_0 - e_1)\big] = e_0.$$

Two "paths" lead to $e_1$ with amplitudes $+\tfrac12$ and $-\tfrac12$: they **cancel**. Two paths lead to $e_0$ with $+\tfrac12$ and $+\tfrac12$: they **add up**. This is impossible with a coin: probabilities cannot subtract, amplitudes can. Every quantum algorithm arranges the computation so that paths to wrong answers cancel and paths to the right one add up.

In [2]:
v1 = H @ e0
v2 = H @ v1
print("after one H: ", np.round(v1, 3), "-> P =", np.round(np.abs(v1) ** 2, 3))
print("after two H's:", np.round(v2, 3), "-> P =", np.round(np.abs(v2) ** 2, 3))

# paths to e1: e0 -> e0 -> e1 and e0 -> e1 -> e1
path_a = H[0, 0] * H[1, 0]
path_b = H[1, 0] * H[1, 1]
print(f"amplitudes of the two paths to e1: {path_a.real:+.2f} and {path_b.real:+.2f}, sum {path_a + path_b:.2f}")

after one H:  [0.707+0.j 0.707+0.j] -> P = [0.5 0.5]
after two H's: [1.+0.j 0.+0.j] -> P = [1. 0.]
amplitudes of the two paths to e1: +0.50 and -0.50, sum 0.00+0.00j

## 5. A circuit is a product of matrices

A quantum circuit (a program) consists of:

1. an initial state, usually $e_0 \otimes \dots \otimes e_0$;
2. a sequence of gates $U_1, U_2, \dots, U_k$;
3. a measurement at the end.

The final state: $v_\text{end} = U_k \cdots U_2 U_1\, v_\text{start}$. **The order is right to left**: the first gate sits next to the vector (circuit diagrams read left to right). The whole circuit without measurement is one unitary matrix $U = U_k \cdots U_1$. An algorithm is a choice of $U$ such that the distribution after measurement is concentrated on the right answer.

A laptop simulator literally multiplies the vector by these matrices. A chip does not: the transpiler turns gates into the processor's native gates, which are executed as microwave pulses.

## 6. Where unitarity comes from physically, and where it ends

**Physics:** a closed quantum system evolves according to the Schrödinger equation, and its solution is always a unitary $U(t)$. For a constant Hamiltonian $\mathcal{H}$ (energy), $U = e^{-i\mathcal{H}t/\hbar}$; for a shaped pulse $\mathcal{H}(t)$ changes in time and $U$ is a time-ordered exponential, still unitary. Unitarity is not a modelling choice; it is how quantum systems evolve.

**Engineering:** a microwave pulse changes the transmon's Hamiltonian for a given time; its shape, length and phase are chosen so that the resulting $U$ is the desired gate.

**Energy:** pulses exchange energy with the qubit in a controlled way. $e_0 \to e_1$ adds a quantum of energy, $e_1 \to e_0$ gives it back, rotations about the $z$ axis do not change populations at all. On IBM chips the $Z$ gate is virtual: software just shifts the phase of subsequent pulses.

**Matrices ↔ physics correspondence:**

- physical operations correspond to unitary matrices **up to a global phase** (the group $PU(2^n) = U(2^n)/U(1)$);
- a small universal gate set approximates any unitary to any accuracy, and the Solovay–Kitaev theorem says this costs only polylog$(1/\varepsilon)$ gates (relevant for discrete error-corrected sets like Clifford+T; IBM's native $R_z$ is continuous); a generic $2^n \times 2^n$ matrix needs exponentially many gates, though, and interesting algorithms are the rare ones that can be built cheaply;
- the action of hardware can be reconstructed by **process tomography**, but only for 1–2 qubits, because the number of measurements grows exponentially;
- real hardware is not a closed system: noise takes us beyond unitary matrices, to quantum channels on density matrices. Hence 0.76 on ibm_fez and 0.76–0.82 on ibm_marrakesh instead of 0.945 from the simulator.

## Summary

> A gate is a unitary matrix: it preserves norms and inner products in every orthonormal basis, it is reversible, and on the Bloch sphere it is a rotation. A circuit is a product of gates (right to left) applied to the initial state, followed by measurement. Physically, gates are microwave pulses whose evolution $e^{-i\mathcal{H}t/\hbar}$ is exactly that matrix, as long as the system is closed.

## Exercises

1. Check that $H$ is unitary: compute $H^{*}H$.
2. Compute $Xu_+$ and $Ze_0$. What do these results say about the rotation axis of each gate?
3. Compute $HZH$. Which familiar gate do you get? Interpret it on the sphere.
4. Why can the matrix $\begin{pmatrix}1 & 1\\ 0 & 0\end{pmatrix}$ ("always set 0") not be a gate?